# Gambeli : mot de réveil « Mésange », version 4

**Pourquoi la v3** : le modèle v1 se réveille sur la radio, la télé et la conversation en français (« mes anges »,
« mise en… »). La v3 apprend avec, comme contre-exemples : de la parole française lue (MLS), de la parole de radio
et de télé (MediaSpeech FR), des mots qui sonnent pareil, et les vrais faux réveils du boîtier (session radio du 07-11).

**Pourquoi la v4** : la v3 ne se réveillait que sur 3 % des « Mésange » propres. Deux des cinq voix de synthèse ne
disent pas le mot : *gilles-low* n'a pas la voyelle nasale (« Missing phoneme from id map ») et babille, *mls-medium*
dit tout autre chose. La v4 n'utilise que les voix qui le disent (siwis, tom, upmc), fait écouter chaque exemple par
Whisper et ne garde que ceux où l'on entend « Mésange », retire « mes anges » des contre-exemples (mêmes sons que le
mot), ne garde que le mot seul (plus de « Mésange, dis-moi »), varie davantage la hauteur de voix, et donne plus de
poids au mot. L'évaluation (cellule 12) est celle de la v3.

**Mode d'emploi** (aussi fait par l'assistant de Chrome, voir CHROME-AGENT.md) :
1. *Exécution › Modifier le type d'exécution* › **GPU T4** › *Enregistrer*.
2. Cellule 3 : mettre le lien de l'export des clips du boîtier dans `NEGATIFS` (ou laisser vide).
3. *Exécution › Tout exécuter*. La cellule 2 redémarre la session **la première fois** : relancer alors *Tout exécuter*.
4. Compter ≈ 3 h 30. Tout est gardé sur Google Drive (dossier `gambeli-mesange-v4`) : après une coupure,
   *Tout exécuter* reprend où cela s'était arrêté (l'entraînement repart de sa dernière étape gardée).
5. À la fin, `mesange-v4.zip` se télécharge (aussi copié sur Drive) : mesange.tflite, mesange.json, metriques-v4.json.

La cellule 12 (évaluation) est la mesure de référence : **ne pas la modifier**.

In [ ]:
%%writefile /content/mesange_v4.py
# 1. Les outils du carnet (mesange_v4.py) : rien à changer ici.
"""Mésange v4 : les outils du carnet d'entraînement (Colab), en Python simple, testables hors Colab.

Le carnet mesange-entrainement-v4.ipynb écrit ce fichier dans /content/mesange_v4.py (cellule 1) et l'importe.
Rien ici n'a besoin du GPU ; les parties « boîtier » (score comme le firmware) ont besoin de pymicro_features
et d'un interpréteur TFLite (ai_edge_litert ou tensorflow), importés seulement quand on s'en sert.

  - patch_microwakeword(dossier) : les trois corrections du code de microWakeWord (idempotent) ;
  - charge16k(chemin) : n'importe quel fichier audio -> float32 mono 16 kHz ;
  - tenu_a_part(nom) : 1 fichier sur 5, choisi par le SHA-1 de son nom, comme le serveur (wakelearn.mjs heldOut) ;
  - lire_export(archive, dossier) : l'export des clips du boîtier (zip, tar ou dossier), trié en positifs/négatifs ;
  - Cache : les étapes coûteuses gardées sur Google Drive, reprises après une coupure ;
  - ModeleBoitier, scores_flux, reveils, meilleur_score : le modèle noté exactement comme le boîtier (main/mww.cc) ;
  - tableau_metriques, seuil_conseille : la table finale 0,90-0,99 ;
  - synthese, rogne, Ecoute, dit_le_mot : les exemples du mot, gardés seulement si la voix a vraiment dit le mot (v4).

Pourquoi la v4 (le modèle v3 ne se réveillait que sur 3 % des « Mésange » propres) : deux des cinq voix ne savent pas
dire le mot. fr_FR-gilles-low n'a pas le signe de la voyelle nasale (« Missing phoneme from id map ») et babille 4 à 11 s ;
fr_FR-mls-medium dit autre chose (Whisper y entend « Merci, merci », « S'il vous plaît »…), quel que soit le locuteur.
40 % des exemples d'apprentissage n'étaient donc pas le mot, et la note finale ne prenait ses « Mésange » que chez
ces voix-là (mls) et le locuteur 0 d'upmc (souvent « Maison »). La v4 retire ces deux voix, écoute chaque exemple
avec Whisper et ne garde que ceux où l'on entend le mot.
"""
import csv, hashlib, io, json, os, re, shutil, subprocess, sys, tarfile, time, unicodedata, urllib.request, wave, zipfile

import numpy as np

RATE = 16000
SEUILS = [0.90, 0.91, 0.92, 0.93, 0.94, 0.95, 0.96, 0.97, 0.98, 0.99]
AUDIO_EXT = ('.wav', '.flac', '.ogg', '.mp3', '.pcm', '.raw', '.opus', '.m4a')


# ---------------------------------------------------------------- audio

def en16k(x, sr):
    from scipy.signal import resample_poly
    x = np.asarray(x, dtype=np.float32)
    if x.ndim > 1: x = x.mean(axis=1)
    if sr != RATE:
        g = np.gcd(int(sr), RATE)
        x = resample_poly(x, RATE // g, int(sr) // g).astype(np.float32)
    return x


def charge16k(chemin):
    """Un fichier audio -> float32 mono 16 kHz dans [-1, 1]. .pcm/.raw : int16 little-endian 16 kHz mono."""
    if chemin.lower().endswith(('.pcm', '.raw')):
        return np.fromfile(chemin, '<i2').astype(np.float32) / 32768
    try:
        import soundfile as sf
        x, sr = sf.read(chemin, dtype='float32', always_2d=False)
    except Exception:
        try:
            with wave.open(chemin, 'rb') as w:
                sr, n, ch, sw = w.getframerate(), w.getnframes(), w.getnchannels(), w.getsampwidth()
                assert sw == 2, f'{chemin}: {8 * sw} bits'
                x = np.frombuffer(w.readframes(n), '<i2').astype(np.float32) / 32768
                if ch > 1: x = x.reshape(-1, ch)
        except Exception:
            import librosa
            x, sr = librosa.load(chemin, sr=None, mono=True)
    return en16k(x, sr)


def ecrit16k(chemin, x):
    """float -> wav 16 kHz 16 bits mono (sans soundfile : le module wave suffit)."""
    pcm = (np.clip(np.asarray(x, np.float32), -1, 1) * 32767).astype('<i2')
    os.makedirs(os.path.dirname(chemin) or '.', exist_ok=True)
    tmp = chemin + '.part'
    with wave.open(tmp, 'wb') as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(RATE); w.writeframes(pcm.tobytes())
    os.replace(tmp, chemin)


def decoupe(x, secondes=20.0, mini=1.0):
    """Coupe un long enregistrement en morceaux de `secondes` au plus (les bouts de moins de `mini` s sautent)."""
    n = int(secondes * RATE)
    return [x[k:k + n] for k in range(0, len(x), n) if len(x[k:k + n]) >= mini * RATE]


def tenu_a_part(nom):
    """1 sur 5, par le SHA-1 du nom de fichier (sans dossier) : la même règle que heldOut() du serveur."""
    return int(hashlib.sha1(os.path.basename(str(nom)).encode()).hexdigest(), 16) % 5 == 0


# ---------------------------------------------------------------- téléchargements

def telecharge(urls, dest, essais=5, attente=10):
    """Télécharge la première adresse qui répond (plusieurs essais chacune) ; reprend un fichier déjà complet."""
    if isinstance(urls, str): urls = [urls]
    if os.path.exists(dest) and os.path.getsize(dest) > 0: return dest
    os.makedirs(os.path.dirname(dest) or '.', exist_ok=True)
    erreurs = []
    for url in urls:
        for k in range(essais):
            try:
                tmp = dest + '.part'
                req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (mesange-v4)'})
                with urllib.request.urlopen(req, timeout=120) as r, open(tmp, 'wb') as f:
                    shutil.copyfileobj(r, f, 1 << 20)
                if os.path.getsize(tmp) == 0: raise IOError('fichier vide')
                os.replace(tmp, dest)
                return dest
            except Exception as e:
                erreurs.append(f'{url} : {e}')
                time.sleep(attente * (k + 1))
    raise IOError('téléchargement impossible :\n  ' + '\n  '.join(erreurs[-6:]))


def lien_direct(url):
    """Un lien de partage Google Drive / Dropbox -> son lien de téléchargement direct ; les autres inchangés."""
    import re
    m = re.search(r'drive\.google\.com/(?:file/d/|open\?id=|uc\?id=)([\w-]+)', url)
    if m: return f'https://drive.usercontent.google.com/download?id={m.group(1)}&export=download&confirm=t'
    if 'dropbox.com' in url: return url.replace('dl=0', 'dl=1') if 'dl=0' in url else url + ('&' if '?' in url else '?') + 'dl=1'
    return url


# ---------------------------------------------------------------- export des clips du boîtier

POSITIFS = {'real', 'missed', 'vrai', 'vrais', 'appel', 'appels', 'positive', 'positives', 'positif', 'positifs', 'wake', 'mot'}


def _etiquettes_index(fichiers):
    """index.json / index.jsonl / index.csv de l'export -> {nom de fichier: étiquette}."""
    etiq = {}
    def ajoute(e):
        if not isinstance(e, dict): return
        f = e.get('file') or e.get('fichier') or e.get('path') or e.get('name')
        lab = e.get('label') or e.get('etiquette') or e.get('kind') or e.get('type')
        if f and lab: etiq[os.path.basename(str(f))] = str(lab).lower()
    for nom, lire in fichiers:
        base = os.path.basename(nom).lower()
        if not base.startswith('index'): continue
        try:
            texte = lire().decode('utf-8', 'replace')
            if base.endswith('.json'):
                d = json.loads(texte)
                for e in (d.get('clips') or d.get('items') or d.get('fichiers') or []) if isinstance(d, dict) else d: ajoute(e)
            elif base.endswith('.jsonl'):
                for ligne in texte.splitlines():
                    if ligne.strip(): ajoute(json.loads(ligne))
            elif base.endswith('.csv'):
                for e in csv.DictReader(io.StringIO(texte)): ajoute(e)
        except Exception as e:
            print('index illisible, ignoré :', nom, e)
    return etiq


def est_positif(nom, etiq):
    lab = etiq.get(os.path.basename(nom))
    if lab is not None: return lab in POSITIFS
    parts = [p.lower() for p in nom.replace('\\', '/').split('/')[:-1]]
    return any(p in POSITIFS for p in parts)


def lire_export(source, dossier):
    """L'export des clips du boîtier -> dossier/{neg_train, neg_test, vrais}/*.wav, et un résumé.

    source : un .zip, un .tar(.gz) ou un dossier. Clips : .wav/.flac/.pcm/.raw... (PCM : int16 16 kHz mono).
    Étiquette : celle de l'index (index.json {clips:[{file,label}]}, .jsonl ou .csv), sinon le nom du dossier.
    « real »/« missed » (vrais appels) -> vrais/ : jamais entraînés, seulement pour la note finale (rappel).
    Le reste (faux réveils, radio...) -> négatifs : 1 sur 5 (tenu_a_part) mis de côté pour la note finale,
    les autres pour l'entraînement, coupés en morceaux de 20 s au plus."""
    fait = os.path.join(dossier, 'resume.json')
    if os.path.exists(fait): return json.load(open(fait))
    tmp = os.path.join(dossier, '_brut')
    shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    if os.path.isdir(source):
        shutil.copytree(source, tmp, dirs_exist_ok=True)
    elif zipfile.is_zipfile(source):
        with zipfile.ZipFile(source) as z: z.extractall(tmp)
    elif tarfile.is_tarfile(source):
        with tarfile.open(source) as t:
            t.extractall(tmp, filter='data') if hasattr(tarfile, 'data_filter') else t.extractall(tmp)
    else:
        raise ValueError(f'{source} : ni zip, ni tar, ni dossier')
    tous = []
    for racine, _, fs in os.walk(tmp):
        for f in fs: tous.append(os.path.join(racine, f))
    etiq = _etiquettes_index([(os.path.relpath(p, tmp), (lambda p=p: open(p, 'rb').read())) for p in tous])
    r = {'vrais': 0, 'neg_train': 0, 'neg_test': 0, 'secondes_neg_test': 0.0, 'secondes_neg_train': 0.0, 'ignores': 0}
    for p in sorted(tous):
        rel = os.path.relpath(p, tmp)
        if not p.lower().endswith(AUDIO_EXT) or os.path.basename(p).startswith('._'): continue
        try:
            x = charge16k(p)
        except Exception as e:
            r['ignores'] += 1; print('clip illisible, ignoré :', rel, e); continue
        if len(x) < RATE // 4: r['ignores'] += 1; continue
        nom = rel.replace('/', '_').replace('\\', '_').rsplit('.', 1)[0]
        if est_positif(rel, etiq):
            ecrit16k(os.path.join(dossier, 'vrais', nom + '.wav'), x); r['vrais'] += 1
        elif tenu_a_part(os.path.basename(p)):
            ecrit16k(os.path.join(dossier, 'neg_test', nom + '.wav'), x)
            r['neg_test'] += 1; r['secondes_neg_test'] += len(x) / RATE
        else:
            for k, morceau in enumerate(decoupe(x, 20.0, 0.5)):
                ecrit16k(os.path.join(dossier, 'neg_train', f'{nom}_{k}.wav'), morceau)
                r['neg_train'] += 1; r['secondes_neg_train'] += len(morceau) / RATE
    shutil.rmtree(tmp, ignore_errors=True)
    json.dump(r, open(fait, 'w'), indent=1)
    return r


# ---------------------------------------------------------------- cache sur Google Drive

class Cache:
    """Chaque étape coûteuse : faite une fois, gardée dans `racine` (Google Drive), reprise à la session suivante.

    c.fait('nom') : l'étape est-elle déjà faite (marque locale ou sur Drive) ?
    c.garde('nom', ['dossier', ...]) : archive les dossiers sur Drive et marque l'étape faite.
    c.reprend('nom') : les dossiers gardés sont-ils là (les restaure depuis Drive au besoin) ?"""

    def __init__(self, racine, local='/content'):
        self.racine, self.local = racine, local
        if racine: os.makedirs(racine, exist_ok=True)

    def _marque(self, nom): return os.path.join(self.local, f'.fait-{nom}')

    def fait(self, nom):
        return os.path.exists(self._marque(nom)) or bool(self.racine and os.path.exists(os.path.join(self.racine, f'{nom}.ok')))

    def garde(self, nom, dossiers=()):
        if self.racine and dossiers:
            tmp = os.path.join(self.racine, f'{nom}.tar.part')
            with tarfile.open(tmp, 'w') as t:
                for d in dossiers: t.add(os.path.join(self.local, d), arcname=d)
            os.replace(tmp, os.path.join(self.racine, f'{nom}.tar'))
        if self.racine: open(os.path.join(self.racine, f'{nom}.ok'), 'w').write(json.dumps(list(dossiers)))
        open(self._marque(nom), 'w').write('ok')

    def reprend(self, nom):
        """True si l'étape est faite et ses dossiers présents ici (restaurés depuis Drive si besoin)."""
        if os.path.exists(self._marque(nom)): return True
        if not self.racine: return False
        ok, arch = os.path.join(self.racine, f'{nom}.ok'), os.path.join(self.racine, f'{nom}.tar')
        if not os.path.exists(ok): return False
        if os.path.exists(arch):
            with tarfile.open(arch) as t:
                t.extractall(self.local, filter='data') if hasattr(tarfile, 'data_filter') else t.extractall(self.local)
        open(self._marque(nom), 'w').write('ok')
        return True


def miroir(src, dst):
    """Copie src -> dst (fichiers nouveaux ou changés seulement)."""
    if not (src and dst and os.path.isdir(src)): return
    for racine, _, fs in os.walk(src):
        cible = os.path.join(dst, os.path.relpath(racine, src)); os.makedirs(cible, exist_ok=True)
        for f in fs:
            a, b = os.path.join(racine, f), os.path.join(cible, f)
            try:
                if not os.path.exists(b) or os.path.getsize(a) != os.path.getsize(b) or os.path.getmtime(a) > os.path.getmtime(b):
                    shutil.copy2(a, b + '.part'); os.replace(b + '.part', b)
            except OSError as e:
                print('miroir : ', a, e)


# ---------------------------------------------------------------- les corrections de microWakeWord

_CLIPS_AVANT = '''        audio_dataset = datasets.Dataset.from_dict(
            {"audio": [str(i) for i in filtered_paths]}
        ).cast_column("audio", datasets.Audio())

        # Convert all clips to 16 kHz sampling rate when accessed
        audio_dataset = audio_dataset.cast_column(
            "audio", datasets.Audio(sampling_rate=16000)
        )
'''
_CLIPS_APRES = '''        # (mesange v3) chemins seuls : l'audio est lu par soundfile (_mesange_charge), pas par datasets/torchcodec
        audio_dataset = datasets.Dataset.from_dict(
            {"audio": [str(i) for i in filtered_paths]}
        )
'''
_CLIPS_FONCTION = '''

def _mesange_charge(path):  # (mesange v3) remplace le décodage de datasets (torchcodec absent ou cassé sur Colab)
    import soundfile as _sf
    from scipy.signal import resample_poly as _rp
    if isinstance(path, dict): path = path.get("path")
    x, sr = _sf.read(path, dtype="float32", always_2d=False)
    if x.ndim > 1: x = x.mean(axis=1)
    if sr != 16000: x = _rp(x, 16000, sr).astype(np.float32)
    return x
'''

_TRAIN_DEBUT_AVANT = '''    best_minimization_quantity = 10000
    best_maximization_quantity = 0.0
    best_no_faph_cutoff = 1.0

    for training_step in range(1, training_steps_max + 1):'''
_TRAIN_DEBUT_APRES = '''    best_minimization_quantity = 10000
    best_maximization_quantity = 0.0
    best_no_faph_cutoff = 1.0

    # (mesange v3) reprise : repartir de la dernière évaluation gardée (resume.json + dernier checkpoint)
    import json as _json, shutil as _shutil
    _resume_path = os.path.join(config["train_dir"], "resume.json")
    _first_step = 1
    if os.path.exists(_resume_path):
        _r = _json.load(open(_resume_path))
        _first_step = int(_r["step"]) + 1
        best_minimization_quantity = _r["best_min"]
        best_maximization_quantity = _r["best_max"]
        best_no_faph_cutoff = _r["best_cutoff"]
        logging.info("Reprise a l'etape %d", _first_step)
        print("REPRISE a l'etape", _first_step, flush=True)

    for training_step in range(_first_step, training_steps_max + 1):'''

_TRAIN_FIN_AVANT = '''            logging.info(
                "So far the best minimization quantity is %.3f with best maximization quantity of %.5f%%; no faph cutoff is %.2f",'''
_TRAIN_FIN_APRES = '''            # (mesange v3) à chaque évaluation : checkpoint + étape, puis copie sur Drive (MESANGE_MIROIR)
            checkpoint.save(file_prefix=checkpoint_prefix)
            with open(_resume_path + ".part", "w") as _f:
                _json.dump({"step": training_step, "best_min": float(best_minimization_quantity), "best_max": float(best_maximization_quantity), "best_cutoff": float(best_no_faph_cutoff)}, _f)
            os.replace(_resume_path + ".part", _resume_path)
            print("ETAPE", training_step, "sur", training_steps_max, "gardee", flush=True)
            if os.environ.get("MESANGE_MIROIR"):
                try:
                    _shutil.copytree(config["train_dir"], os.environ["MESANGE_MIROIR"], dirs_exist_ok=True)
                except Exception as _e:
                    print("miroir Drive impossible :", _e, flush=True)

            logging.info(
                "So far the best minimization quantity is %.3f with best maximization quantity of %.5f%%; no faph cutoff is %.2f",'''


def _remplace(chemin, avant, apres, deja):
    texte = open(chemin, encoding='utf-8').read()
    if deja in texte: return 'déjà'
    assert texte.count(avant) == 1, f'{chemin} : passage attendu introuvable (microWakeWord a changé ?)'
    open(chemin, 'w', encoding='utf-8').write(texte.replace(avant, apres))
    return 'corrigé'


def patch_microwakeword(racine):
    """Les corrections du code de microWakeWord (commit épinglé), sans effet si déjà faites. Renvoie un résumé."""
    pkg = os.path.join(racine, 'microwakeword')
    r = {}
    # 1. data.py : validation ambiante 3 fois moins dense (sinon plus de mémoire vive : plantage du 31-10)
    r['data'] = _remplace(os.path.join(pkg, 'data.py'), '                    int(1000 * self.step * self.stride),\n',
                          '                    3 * int(1000 * self.step * self.stride),\n', '3 * int(1000 * self.step * self.stride)')
    # 2. clips.py : lire l'audio avec soundfile, plus avec datasets (torchcodec : « AudioDecoder not subscriptable »)
    c = os.path.join(pkg, 'audio', 'clips.py')
    r['clips'] = _remplace(c, _CLIPS_AVANT, _CLIPS_APRES, '(mesange v3) chemins seuls')
    texte = open(c, encoding='utf-8').read()
    if '_mesange_charge(path)' not in texte:
        assert texte.count('\nclass Clips') == 1, f'{c} : classe Clips introuvable'
        texte = texte.replace('\nclass Clips', _CLIPS_FONCTION + '\n\nclass Clips', 1)
    texte = texte.replace('clip["audio"]["array"]', '_mesange_charge(clip["audio"])')
    texte = texte.replace('rand_audio_entry["audio"]["array"]', '_mesange_charge(rand_audio_entry["audio"])')
    assert '["audio"]["array"]' not in texte, f'{c} : un accès audio non corrigé'
    open(c, 'w', encoding='utf-8').write(texte)
    # 3. train.py : reprendre à la dernière évaluation après une coupure (et copier sur Drive)
    t = os.path.join(pkg, 'train.py')
    r['train_debut'] = _remplace(t, _TRAIN_DEBUT_AVANT, _TRAIN_DEBUT_APRES, '(mesange v3) reprise')
    r['train_fin'] = _remplace(t, _TRAIN_FIN_AVANT, _TRAIN_FIN_APRES, '(mesange v3) à chaque évaluation')
    for f in ('data.py', 'audio/clips.py', 'train.py'):
        compile(open(os.path.join(pkg, f), encoding='utf-8').read(), f, 'exec')
    return r


# ---------------------------------------------------------------- le modèle noté comme le boîtier

FEATURES, FENETRE = 40, 5
RECUL = FENETRE * 3       # après un réveil, le firmware ignore 15 inférences (d.ignore = -WINDOW * 3)


def caracteristiques(pcm_int16):
    """Le frontend du boîtier (TFLite Micro, 40 bandes, comme ESPHome), quantifié en int8 comme feed() de mww.cc."""
    import pymicro_features
    fe, sortie, buf, at = pymicro_features.MicroFrontend(), [], np.asarray(pcm_int16, '<i2').tobytes(), 0
    while at + 320 <= len(buf):
        r = fe.process_samples(buf[at:at + 320])
        at += r.samples_read * 2
        if r.features:
            v = np.rint(np.asarray(r.features) * 25.6).astype(np.int32)
            sortie.append(np.clip((v * 256 + 333) // 666 - 128, -128, 127).astype(np.int8))
        if r.samples_read == 0: break
    return np.array(sortie, np.int8).reshape(-1, FEATURES)


def en_int16(x):
    x = np.asarray(x)
    return x if x.dtype == np.int16 else (np.clip(x, -1, 1) * 32767).astype(np.int16)


class ModeleBoitier:
    """Un modèle microWakeWord en flux (.tflite quantifié), appelé comme sur le boîtier."""

    def __init__(self, chemin):
        try:
            from ai_edge_litert.interpreter import Interpreter
        except Exception:
            from tensorflow.lite import Interpreter
        self.it = Interpreter(model_path=chemin, num_threads=1)
        self.it.allocate_tensors()
        i = self.it.get_input_details()[0]
        self.inp, self.out = i['index'], self.it.get_output_details()[0]['index']
        self.stride = int(i['shape'][1])
        assert i['dtype'] == np.int8 and i['shape'][2] == FEATURES, "ce n'est pas un modèle microWakeWord en flux"

    def scores(self, feats):
        """Sortie uint8 de chaque inférence, état remis à zéro au départ (comme après chaque échange)."""
        self.it.reset_all_variables()
        out = []
        for k in range(0, len(feats) - self.stride + 1, self.stride):
            self.it.set_tensor(self.inp, feats[k:k + self.stride][None])
            self.it.invoke()
            out.append(int(self.it.get_tensor(self.out).reshape(-1)[0]))
        return np.array(out, np.int32)


def scores_flux(modele, pcm, morceau_s=600):
    """Scores d'un long enregistrement, par morceaux de 10 min (mémoire bornée)."""
    pcm, n, out = en_int16(pcm), int(morceau_s * RATE), []
    for k in range(0, len(pcm), n):
        out.append(modele.scores(caracteristiques(pcm[k:k + n])))
    return np.concatenate(out) if out else np.zeros(0, np.int32)


def sommes_fenetre(s):
    p = np.concatenate([np.zeros(FENETRE - 1, np.int32), np.asarray(s, np.int32)])
    return np.convolve(p, np.ones(FENETRE, np.int32), 'valid')


def meilleur_score(s):
    """La meilleure moyenne sur 5 inférences (0..1) : le clip réveille le boîtier au seuil c si elle le dépasse."""
    return int(sommes_fenetre(s).max()) / FENETRE / 255 if len(s) else 0.0


def note_clip(modele, chemin, marge_s=0.5):
    """Le meilleur score d'un clip du mot, entouré de `marge_s` secondes de silence (sur le boîtier, il y a
    toujours du son avant le mot, et le modèle répond un peu après la fin du mot)."""
    z = np.zeros(int(marge_s * RATE), np.float32)
    return meilleur_score(modele.scores(caracteristiques(en_int16(np.concatenate([z, charge16k(chemin), z])))))


def reveille(meilleur, seuil):
    """Même comparaison que le firmware : somme > int(seuil*255)*5."""
    return round(meilleur * FENETRE * 255) > int(seuil * 255) * FENETRE


def reveils(s, seuil):
    """Nombre de réveils sur un long flux, avec la remise à zéro et le recul du firmware."""
    barre, recent, n, ignore, last = int(seuil * 255) * FENETRE, [0] * FENETRE, 0, 0, 0
    for x in s:
        last = (last + 1) % FENETRE; recent[last] = int(x)
        if ignore < 0: ignore += 1; continue
        if sum(recent) > barre: n += 1; recent = [0] * FENETRE; ignore = -RECUL
    return n


def flux_de_dossier(fichiers, blanc_s=0.3, max_s=None):
    """Les fichiers mis bout à bout (un petit silence entre deux), en int16 ; s'arrête à max_s secondes."""
    blanc, parts, total = np.zeros(int(blanc_s * RATE), np.int16), [], 0
    for f in fichiers:
        try: x = en_int16(charge16k(f))
        except Exception as e: print('ignoré', f, e); continue
        parts += [x, blanc]; total += len(x) + len(blanc)
        if max_s and total >= max_s * RATE: break
    return np.concatenate(parts) if parts else np.zeros(0, np.int16)


# ---------------------------------------------------------------- la table finale

def tableau_metriques(fa_sources, rappels, seuils=SEUILS):
    """fa_sources : {source: {'heures': h, 'scores': scores du flux}} ; rappels : {nom: [meilleur score de chaque clip]}.
    -> une ligne par seuil : réveils/heure par source, sur toute la parole française (« parole_fr » : les sources
    dont le nom commence par fr_), et le rappel (part des clips du mot qui réveillent)."""
    lignes = []
    for c in seuils:
        l = {'seuil': round(c, 2), 'fa_h': {}, 'rappel': {}}
        n_fr = h_fr = 0
        for src, d in fa_sources.items():
            n = reveils(d['scores'], c)
            l['fa_h'][src] = round(n / d['heures'], 3) if d['heures'] else None
            if src.startswith('fr_'): n_fr += n; h_fr += d['heures']
        l['fa_h']['parole_fr'] = round(n_fr / h_fr, 3) if h_fr else None
        for nom, best in rappels.items():
            l['rappel'][nom] = round(sum(reveille(b, c) for b in best) / len(best), 4) if len(best) else None
        lignes.append(l)
    return lignes


def seuil_conseille(lignes, fa_max=0.5):
    """Parmi les seuils à au plus `fa_max` réveil/heure sur la parole française, celui qui garde le plus de vrais
    appels (s'il y en a), puis le meilleur rappel « bruit », puis le plus bas ; aucun : celui au moins de faux
    réveils (à égalité : le plus bas)."""
    ok = [l for l in lignes if l['fa_h'].get('parole_fr') is not None and l['fa_h']['parole_fr'] <= fa_max]
    if ok:
        return max(ok, key=lambda l: (l['rappel'].get('vrais_appels') or 0, l['rappel'].get('bruit') or 0, -l['seuil']))['seuil']
    return min(lignes, key=lambda l: (l['fa_h'].get('parole_fr') or 0, l['seuil']))['seuil']


def texte_tableau(lignes):
    """La table en Markdown, pour l'affichage et le rapport."""
    sources = [k for k in lignes[0]['fa_h']] if lignes else []
    rappels = [k for k in lignes[0]['rappel']] if lignes else []
    tete = ['seuil'] + [f'réveils/h {s}' for s in sources] + [f'rappel {r}' for r in rappels]
    out = ['| ' + ' | '.join(tete) + ' |', '|' + '---|' * len(tete)]
    for l in lignes:
        cases = [f"{l['seuil']:.2f}"] + ['–' if l['fa_h'][s] is None else f"{l['fa_h'][s]:.2f}" for s in sources]
        cases += ['–' if l['rappel'][r] is None else f"{100 * l['rappel'][r]:.1f} %" for r in rappels]
        out.append('| ' + ' | '.join(cases) + ' |')
    return '\n'.join(out)


def versions(paquets=('tensorflow', 'numpy', 'keras', 'audiomentations', 'datasets', 'piper-tts', 'pymicro-features',
                      'ai-edge-litert', 'mmap-ninja', 'soundfile', 'scipy', 'protobuf')):
    from importlib import metadata
    r = {}
    for p in paquets:
        try: r[p] = metadata.version(p)
        except Exception: r[p] = None
    try: r['gpu'] = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True, timeout=20).stdout.strip() or None
    except Exception: r['gpu'] = None
    return r


# ---------------------------------------------------------------- commandes, voix, spectrogrammes (Colab)

def sh(cmd, flux=False, essais=1):
    """Une commande shell ; sa sortie s'affiche. En cas d'échec : une erreur avec la fin de la sortie (pour corriger)."""
    for k in range(essais):
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        fin = []
        for ligne in p.stdout:
            fin = (fin + [ligne.rstrip()])[-60:]
            if flux: print(ligne, end='', flush=True)
        if p.wait() == 0:
            if not flux and fin: print('\n'.join(fin[-5:]))
            return
        if k + 1 < essais: print(f'échec, nouvel essai ({k + 2}/{essais})…'); time.sleep(15)
    raise RuntimeError(f'commande en échec (code {p.returncode}) : {cmd}\n' + '\n'.join(fin))


def voix_piper(dossier, voix, base='https://huggingface.co/rhasspy/piper-voices/resolve/main/'):
    os.makedirs(dossier, exist_ok=True)
    for chemin, nom in voix:
        for ext in ('.onnx', '.onnx.json'):
            telecharge(f'{base}{chemin}/{nom}{ext}', os.path.join(dossier, nom + ext))


def phonemes_manquants(voix, texte):
    """Les phonèmes du texte que la voix Piper ne connaît pas (« Missing phoneme from id map ») : [] si elle sait le dire."""
    return sorted({p for phrase in voix.phonemize(texte) for p in phrase if p not in voix.config.phoneme_id_map})


def rogne(x, seuil_db=-40.0, marge_s=0.05):
    """Retire le silence avant et après la parole (trames de 10 ms à plus de `seuil_db` sous le maximum), en gardant
    `marge_s` de chaque côté : le mot finit alors toujours à la même distance de la fin de la fenêtre d'apprentissage."""
    x = np.asarray(x, np.float32)
    n = len(x) // 160
    if n == 0: return x
    e = 10 * np.log10((x[:n * 160].reshape(n, 160) ** 2).mean(axis=1) + 1e-12)
    actif = np.flatnonzero(e > e.max() + seuil_db)
    m = int(marge_s * RATE)
    return x[max(0, actif[0] * 160 - m):min(len(x), (actif[-1] + 1) * 160 + m)]


def lettres(texte):
    """« Mé-sange ! » -> « mesange » : minuscules, sans accents, lettres seules."""
    t = unicodedata.normalize('NFKD', str(texte).lower()).encode('ascii', 'ignore').decode()
    return re.sub(r'[^a-z]', '', t)


# Ce que Whisper écrit quand il entend [mezɑ̃ʒ] : Mésange, Maisange, Mes anges, Messange, Mésanche...
MOT_ENTENDU = re.compile(r'^m(e|ai|ei)(s|ss|z)an(g|j|ch)(e|es)?$')


def dit_le_mot(transcription, motif=MOT_ENTENDU):
    """La transcription n'est-elle que le mot (rien avant, rien après) ?"""
    return bool(motif.match(lettres(transcription)))


def cuda12():
    """Colab (11-2026) n'a plus libcublas.so.12 ni cuDNN 9, que faster-whisper charge sur GPU : pris de pip, mis
    dans le dossier standard des bibliothèques (lu à chaque chargement). Sans GPU ou sans droit : ne fait rien."""
    import glob, shutil, site, subprocess
    if glob.glob('/usr/lib/x86_64-linux-gnu/libcublas.so.12*') or not os.path.exists('/usr/lib/x86_64-linux-gnu'):
        return
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'nvidia-cublas-cu12', 'nvidia-cudnn-cu12==9.*'], check=False)
    for d in site.getsitepackages():
        for f in glob.glob(d + '/nvidia/cublas/lib/*.so*') + glob.glob(d + '/nvidia/cudnn/lib/*.so*'):
            try:
                shutil.copy(f, '/usr/lib/x86_64-linux-gnu/')
            except OSError:
                return
    subprocess.run(['ldconfig'], check=False)


class Ecoute:
    """Whisper (faster-whisper) en français : vérifie qu'un exemple de synthèse dit bien le mot. GPU si possible."""

    def __init__(self, taille='small'):
        from faster_whisper import WhisperModel   # noms complets : un dossier « small » ou « base » ici serait pris pour le modèle
        try:
            cuda12()
            self.m = WhisperModel(f'Systran/faster-whisper-{taille}', device='cuda', compute_type='float16')
            self.texte(np.zeros(RATE, np.float32))   # la bibliothèque GPU n'est chargée qu'au premier passage
        except Exception as e:
            print('Whisper sans GPU (', e, ') : modèle « base » sur le processeur')
            self.m = WhisperModel('Systran/faster-whisper-base', device='cpu', compute_type='int8')

    def texte(self, x):
        z = np.zeros(RATE // 2, np.float32)
        segs, _ = self.m.transcribe(np.concatenate([z, np.asarray(x, np.float32), z]), language='fr', beam_size=1,
                                    condition_on_previous_text=False, vad_filter=False)
        return ''.join(s.text for s in segs).strip()

    def __call__(self, x):
        return dit_le_mot(self.texte(x))


def synthese(dossier_voix, voix, textes, n, dossier, graine, eval_seulement=False, ecoute=None, duree_max=2.0, essais=8):
    """n clips des `textes` par les voix Piper, en wav 16 kHz dans `dossier` (reprend où il en était).
    Locuteurs des voix à plusieurs locuteurs : 1 sur 5 (id % 5 == 0) réservé à l'évaluation ;
    eval_seulement=True ne prend que ceux-là (et seulement ces voix-là).
    v4 : une voix qui ne connaît pas un phonème des textes est refusée ; chaque clip est rogné (silences) ; avec
    `ecoute` (Ecoute()), un clip n'est gardé que si l'on y entend le mot et s'il dure au plus `duree_max` s ;
    au plus `essais` fois n tentatives. Le compte des refus est écrit dans dossier/refus.json."""
    import random
    from piper import PiperVoice, SynthesisConfig
    os.makedirs(dossier, exist_ok=True)
    deja = len([f for f in os.listdir(dossier) if f.endswith('.wav')])
    if deja >= n: return deja
    rnd = random.Random(graine * 1000 + deja)
    charges = []
    for _, nom in voix:
        v = PiperVoice.load(os.path.join(dossier_voix, nom + '.onnx'))
        manque = sorted({p for t in textes for p in phonemes_manquants(v, t)})
        assert not manque, f'{nom} ne sait pas dire {textes[0]!r} (phonèmes absents : {manque}) : retirer cette voix'
        loc = v.config.num_speakers or 1
        ids = [s for s in range(loc) if (s % 5 == 0) == eval_seulement] if loc > 1 else ([None] if not eval_seulement else [])
        if ids: charges.append((nom, v, ids))
    assert charges, 'aucune voix utilisable'
    refus = {nom: 0 for nom, _, _ in charges}
    k, tentatives = deja, 0
    while k < n:
        tentatives += 1
        assert tentatives <= essais * n, f'trop de refus ({refus}) : les voix ne disent pas le mot'
        nom, v, ids = charges[tentatives % len(charges)]
        cfg = SynthesisConfig(speaker_id=rnd.choice(ids), length_scale=rnd.uniform(0.75, 1.25),
                              noise_scale=rnd.uniform(0.4, 1.0), noise_w_scale=rnd.uniform(0.5, 1.0))
        audio = b''.join(c.audio_int16_bytes for c in v.synthesize(rnd.choice(textes), syn_config=cfg))
        x = rogne(en16k(np.frombuffer(audio, np.int16).astype(np.float32) / 32768, v.config.sample_rate))
        if ecoute is not None and (len(x) > duree_max * RATE or not ecoute(x)):
            refus[nom] += 1; continue
        ecrit16k(os.path.join(dossier, f'{k:05d}.wav'), x)
        k += 1
    json.dump({'gardes': k - deja, 'tentatives': tentatives, 'refus': refus}, open(os.path.join(dossier, 'refus.json'), 'w'))
    print(dossier, ':', k - deja, 'gardés sur', tentatives, '; refusés par voix :', refus)
    return k


def augmenteur(fonds, rirs, duree=3.2, snr=(-5, 10), proba_fond=0.75, proba_hauteur=0.1):
    from microwakeword.audio.augmentation import Augmentation
    return Augmentation(augmentation_duration_s=duree,
                        augmentation_probabilities={"SevenBandParametricEQ": 0.1, "TanhDistortion": 0.1, "PitchShift": proba_hauteur,
                                                    "BandStopFilter": 0.1, "AddColorNoise": 0.1, "AddBackgroundNoise": proba_fond,
                                                    "Gain": 1.0, "RIR": 0.5},
                        impulse_paths=rirs, background_paths=fonds,
                        background_min_snr_db=snr[0], background_max_snr_db=snr[1],
                        min_jitter_s=0.195, max_jitter_s=0.205)


def spectrogrammes(source, cible, augmenter, positif=False, nom_mmap='neg_mmap'):
    """Les spectrogrammes d'un dossier de wav, rangés comme microWakeWord les attend (training/validation/testing).
    Refait l'étape entière si elle avait été coupée (marque cible/.ok)."""
    from mmap_ninja.ragged import RaggedMmap
    from microwakeword.audio.clips import Clips
    from microwakeword.audio.spectrograms import SpectrogramGeneration
    if os.path.exists(os.path.join(cible, '.ok')): return 'déjà'
    shutil.rmtree(cible, ignore_errors=True)
    clips = Clips(input_directory=source, file_pattern='*.wav', max_clip_duration_s=None, remove_silence=False,
                  random_split_seed=10, split_count=0.1)
    for split, nom in [('training', 'train'), ('validation', 'validation'), ('testing', 'test')]:
        if positif:
            gen = SpectrogramGeneration(clips=clips, augmenter=augmenter, slide_frames=1 if split == 'testing' else 10, step_ms=10)
            rep = 2 if split == 'training' else 1
        else:
            gen = SpectrogramGeneration(clips=clips, augmenter=augmenter, slide_frames=1, step_ms=10)
            rep = 1
        os.makedirs(os.path.join(cible, split), exist_ok=True)   # mmap_ninja ne crée que le dernier dossier
        RaggedMmap.from_generator(out_dir=os.path.join(cible, split, nom_mmap),
                                  sample_generator=gen.spectrogram_generator(split=nom, repeat=rep),
                                  batch_size=100, verbose=True)
    open(os.path.join(cible, '.ok'), 'w').write('ok')
    return 'fait'

In [ ]:
# 2. Installation (une seule fois par session Colab) -- la session redémarre ensuite d'elle-même : c'est normal,
#    relancez alors « Tout exécuter ». Versions épinglées : celles qui ont tourné sur Colab le 31-10.
import os, sys, subprocess
sys.path.insert(0, '/content'); os.chdir('/content')
from mesange_v4 import sh, patch_microwakeword, versions
MWW_COMMIT = '4665173cd35f1cff9a61e06fc427f124766c488e'     # microWakeWord, « Pin dependencies (#98) »
PAQUETS = ["piper-tts==1.3.0", "pymicro-features==2.0.2", "ai-edge-litert==2.3.0", "audiomentations==0.43.1",
           "mmap-ninja==0.9.0", "webrtcvad-wheels==2.0.14.post1", "soundfile", "pyyaml", "librosa", "faster-whisper==1.2.1",
           "git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f"]
VERIF = ("import tensorflow as tf, numpy, microwakeword, pymicro_features, audiomentations, mmap_ninja, soundfile, yaml; "
         "from microwakeword.audio.clips import Clips; from microwakeword.audio.spectrograms import SpectrogramGeneration; "
         "from piper import PiperVoice, SynthesisConfig; from ai_edge_litert.interpreter import Interpreter; "
         "from faster_whisper import WhisperModel; "
         "print('tensorflow', tf.__version__, 'numpy', numpy.__version__, 'GPU', len(tf.config.list_physical_devices('GPU')))")

def verifie():
    r = subprocess.run([sys.executable, '-c', VERIF], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip()[-3000:])
    return r.returncode == 0, r.stdout

if os.path.exists('/content/.installed-v4'):
    print('Déjà installé : on continue.')
else:
    if not os.path.isdir('microWakeWord/.git'):
        sh('git clone -q https://github.com/kahrendt/microWakeWord', essais=3)
    sh(f'git -C microWakeWord fetch -q origin && git -C microWakeWord checkout -q -f {MWW_COMMIT}', essais=3)
    print(patch_microwakeword('/content/microWakeWord'))
    sh('pip install -q --no-deps -e ./microWakeWord', essais=2)
    sh('pip install -q ' + ' '.join(f'"{p}"' for p in PAQUETS), essais=3)
    ok, sortie = verifie()
    if not ok:   # secours : la TensorFlow connue pour aller avec microWakeWord (2.21) et son protobuf
        sh('pip install -q "tensorflow==2.21.0" "protobuf>=6.31.1,<8" "numpy>=2.0,<2.4"', essais=2)
        ok, sortie = verifie()
    assert ok, "Les bibliothèques ne s'importent pas (voir l'erreur au-dessus) : corriger PAQUETS puis relancer."
    if 'GPU 0' in sortie:
        print("ATTENTION : pas de GPU. Exécution › Modifier le type d'exécution › GPU T4, puis tout relancer.")
        raise SystemExit('pas de GPU')
    open('/content/.installed-v4', 'w').write('ok')
    print('Installé. La session redémarre (normal) : relancez « Tout exécuter ».')
    os.kill(os.getpid(), 9)

In [ ]:
# 3. Paramètres, Google Drive (pour reprendre après une coupure) et l'export des clips du boîtier.
import os, sys, json, glob
sys.path.insert(0, '/content'); os.chdir('/content')
from mesange_v4 import *

MOT, NOM = 'Mésange', 'mesange'
NEGATIFS = ''          # lien de l'export des clips du boîtier (zip) ; vide : un fichier déposé dans /content
                       # (negatifs*.zip) ou sur Drive (gambeli-mesange-v4/negatifs.zip) ; sinon on s'en passe.
ETAPES = 15000         # étapes d'entraînement (v2 : 15000)

assert os.path.exists('/content/.installed-v4'), "Lancez d'abord la cellule 2."
drive = ''
try:
    from google.colab import drive as _d
    _d.mount('/content/drive')
    drive = '/content/drive/MyDrive/gambeli-mesange-v4'
    os.makedirs(drive, exist_ok=True)
except Exception as e:
    print('Google Drive non monté (', e, ') : pas de reprise possible après une coupure, on continue quand même.')

export = ''
if NEGATIFS.strip().startswith('http'):
    export = telecharge(lien_direct(NEGATIFS.strip()), '/content/negatifs.zip')
else:
    trouves = sorted(glob.glob('/content/negatifs*.zip') + glob.glob('/content/*export*.zip') + ([f'{drive}/negatifs.zip'] if drive else []))
    trouves = [f for f in trouves if os.path.exists(f)]
    export = trouves[0] if trouves else ''
if export and drive and not os.path.exists(f'{drive}/negatifs.zip'):
    shutil.copy(export, f'{drive}/negatifs.zip')
boitier = lire_export(export, '/content/boitier') if export else None
print('Export du boîtier :', boitier or 'aucun (le carnet tourne sans)')

P = dict(mot=MOT, nom=NOM, etapes=ETAPES, drive=drive, export=export, boitier=boitier)
json.dump(P, open('/content/params.json', 'w'), indent=1)
print(json.dumps(P, ensure_ascii=False, indent=1))

In [ ]:
import os, sys, json, glob, shutil, tempfile
import numpy as np
sys.path.insert(0, '/content'); os.chdir('/content')
from mesange_v4 import *
P = json.load(open('/content/params.json'))   # écrit par la cellule 3
cache = Cache(P['drive'])

# 4. Le mot dit par des voix françaises de synthèse (Piper) : 3000 pour apprendre ; 400 par des locuteurs
#    jamais entendus à l'entraînement, pour la note finale ; et les mots qui sonnent pareil (contre-exemples).
#    v4 : seulement les voix qui savent dire le mot (gilles-low n'a pas la voyelle nasale de « -sange », mls-medium
#    dit autre chose) ; chaque exemple du mot est écouté par Whisper et gardé seulement si l'on y entend « Mésange »
#    (upmc le réussit environ 1 fois sur 8 : c'est normal que beaucoup d'essais soient refusés).
import gc
VOIX = [('fr/fr_FR/siwis/medium', 'fr_FR-siwis-medium'), ('fr/fr_FR/tom/medium', 'fr_FR-tom-medium'),
        ('fr/fr_FR/upmc/medium', 'fr_FR-upmc-medium')]
MOT = P['mot']
# Le mot seul : la fenêtre d'apprentissage finit 0,2 s après lui (avec « Mésange, dis-moi », elle finirait sur « dis-moi »).
DITES = [MOT, MOT + ' !', MOT + ' ?', MOT + '.', 'Mé-sange', MOT.lower()]
# Ce que le boîtier a pris pour le mot (07-11) et ce qui sonne comme lui, sans l'être.
# Pas « mes anges » : mêmes sons que « Mésange » ([mezɑ̃ʒ]) ; l'apprendre comme contre-exemple contredit le mot.
VOISINS = ['les anges', 'des anges', 'mais sans', 'mes sangles', 'mélange', 'le mélange', 'mésentente',
           'message', 'messages', 'méchant', 'mes enfants', 'les enfants', 'des ans', 'mais en', 'mais encore',
           'mise en œuvre', 'mise en jeu', 'mise en route', 'mise en place', 'mes doigts', 'les ongles', 'mes ongles',
           'mais si', 'mais ça change', 'des oranges', 'des mésaventures', 'mais je mange', 'massage', 'ma sauge',
           'mes amis', 'mais attends', 'mes affaires', 'les échanges', 'mes ennemis', 'messagerie', 'mais enfin']

if cache.reprend('voix'):
    print('voix et exemples repris de Drive')
else:
    voix_piper('voix', VOIX)
    ecoute = Ecoute()
    print(synthese('voix', VOIX, DITES, 3000, 'generated_samples', graine=1, ecoute=ecoute), 'exemples du mot (apprentissage)')
    print(synthese('voix', VOIX, DITES, 400, 'eval_positifs', graine=2, eval_seulement=True, ecoute=ecoute, essais=30),
          'exemples du mot (note finale)')
    del ecoute; gc.collect()
    print(synthese('voix', VOIX, VOISINS, 1600, 'near_words', graine=3), 'mots voisins')
    cache.garde('voix', ['voix', 'generated_samples', 'eval_positifs', 'near_words'])
print(len(glob.glob('generated_samples/*.wav')), len(glob.glob('eval_positifs/*.wav')), len(glob.glob('near_words/*.wav')))

In [ ]:
import os, sys, json, glob, shutil, tempfile
import numpy as np
sys.path.insert(0, '/content'); os.chdir('/content')
from mesange_v4 import *
P = json.load(open('/content/params.json'))   # écrit par la cellule 3
cache = Cache(P['drive'])

# 5. Bruits et pièces pour « salir » les exemples : réponses impulsionnelles MIT, AudioSet (1 tranche), FMA (musique).
#    (Corrections du 31-10 reprises : lecture par soundfile, AudioSet en parquet, RIR par snapshot_download.)
import io, tempfile
from pathlib import Path
from tqdm import tqdm

if cache.reprend('bruits'):
    print('bruits repris de Drive')
else:
    if len(glob.glob('mit_rirs/*.wav')) < 100:
        from huggingface_hub import snapshot_download
        src = snapshot_download('davidscripka/MIT_environmental_impulse_responses', repo_type='dataset')
        for p in tqdm([p for p in Path(src).rglob('*') if p.suffix.lower() in ('.wav', '.flac', '.mp3', '.ogg')]):
            try: ecrit16k(f'mit_rirs/{p.stem}.wav', charge16k(str(p)))
            except Exception as e: print('ignoré', p.name, e)
    if len(glob.glob('audioset_16k/*.wav')) < 100:
        import pyarrow.parquet as pq
        pqf = telecharge('https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train/09.parquet', 'audioset/09.parquet')
        k = 0
        for batch in pq.ParquetFile(pqf).iter_batches(batch_size=64, columns=['audio']):
            for a in batch.column('audio').to_pylist():
                k += 1
                try:
                    with tempfile.NamedTemporaryFile(suffix=os.path.splitext(a.get('path') or 'x.flac')[1] or '.flac') as t:
                        t.write(a['bytes']); t.flush(); ecrit16k(f'audioset_16k/{k}.wav', charge16k(t.name))
                except Exception as e:
                    if k < 5: print('ignoré', k, e)
        os.remove(pqf)
    if len(glob.glob('fma_16k/*.wav')) < 100:
        z = telecharge('https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip', 'fma/fma_xs.zip')
        sh('cd fma && unzip -q -o fma_xs.zip')
        for p in tqdm(list(Path('fma').glob('**/*.mp3'))):
            out = f'fma_16k/{p.stem}.wav'
            if not os.path.exists(out):
                try: ecrit16k(out, charge16k(str(p)))
                except Exception as e: print('ignoré', p.name, e)
        shutil.rmtree('fma', ignore_errors=True)
    cache.garde('bruits', ['mit_rirs', 'audioset_16k', 'fma_16k'])
print(len(os.listdir('mit_rirs')), 'RIR,', len(os.listdir('audioset_16k')), 'audioset,', len(os.listdir('fma_16k')), 'fma')

In [ ]:
import os, sys, json, glob, shutil, tempfile
import numpy as np
sys.path.insert(0, '/content'); os.chdir('/content')
from mesange_v4 import *
P = json.load(open('/content/params.json'))   # écrit par la cellule 3
cache = Cache(P['drive'])

# 6. Les exemples du mot, « salis » (bruit, pièce, distorsion), en spectrogrammes pour l'entraînement.
#    v4 : hauteur de voix changée une fois sur deux (v3 : 1 sur 10), pour d'autres voix que les trois de synthèse.
aug = augmenteur(['fma_16k', 'audioset_16k'], ['mit_rirs'], proba_hauteur=0.5)
print(spectrogrammes('generated_samples', 'generated_augmented_features', aug, positif=True, nom_mmap='wakeword_mmap'))

In [ ]:
import os, sys, json, glob, shutil, tempfile
import numpy as np
sys.path.insert(0, '/content'); os.chdir('/content')
from mesange_v4 import *
P = json.load(open('/content/params.json'))   # écrit par la cellule 3
cache = Cache(P['drive'])

# 7. Contre-exemples tout prêts de microWakeWord (parole anglaise, dîner, bruits sans parole).
os.makedirs('negative_datasets', exist_ok=True)
for f in ['dinner_party.zip', 'dinner_party_eval.zip', 'no_speech.zip', 'speech.zip']:
    marque = f'negative_datasets/.{f}.ok'
    if os.path.exists(marque): continue
    z = telecharge('https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/' + f, f'dl/{f}')
    sh(f'unzip -q -o {z} -d negative_datasets')
    os.remove(z); open(marque, 'w').write('ok')
print(sorted(os.listdir('negative_datasets')))

In [ ]:
import os, sys, json, glob, shutil, tempfile
import numpy as np
sys.path.insert(0, '/content'); os.chdir('/content')
from mesange_v4 import *
P = json.load(open('/content/params.json'))   # écrit par la cellule 3
cache = Cache(P['drive'])

# 8. Contre-exemples FRANÇAIS (le cœur de la v4) :
#  a) parole lue : MLS français (CC-BY 4.0), ≈ 15 h, locuteurs d'apprentissage ;
#  b) radio et télé : MediaSpeech FR (CC-BY 4.0, ≈ 10 h d'émissions) : 4 fichiers sur 5 (le 5e sert à la note finale) ;
#  c) les mots voisins (cellule 4) ; d) les faux réveils du boîtier (export, cellule 3) : 4 sur 5.
import random, tarfile
from tqdm import tqdm
from huggingface_hub import list_repo_files, hf_hub_download
MLS = 'facebook/multilingual_librispeech'

if not os.path.exists('fr_speech/.ok'):
    os.makedirs('fr_speech', exist_ok=True)
    fichiers = sorted(f for f in list_repo_files(MLS, repo_type='dataset') if f.startswith('data/mls_french/train/audio/'))
    random.Random(2).shuffle(fichiers)
    for f in tqdm(fichiers[:60]):
        p = hf_hub_download(MLS, f, repo_type='dataset', local_dir='mls_dl')
        with tarfile.open(p) as t:
            for m in t.getmembers():
                if m.isfile() and m.name.lower().endswith(('.flac', '.wav', '.opus')):
                    nom = os.path.basename(m.name).rsplit('.', 1)[0]
                    with tempfile.NamedTemporaryFile(suffix=os.path.splitext(m.name)[1]) as tmp:
                        tmp.write(t.extractfile(m).read()); tmp.flush()
                        ecrit16k(f'fr_speech/{nom}.wav', charge16k(tmp.name))
        os.remove(p)
    open('fr_speech/.ok', 'w').write('ok')
print(len(os.listdir('fr_speech')), 'enregistrements MLS')

if not os.path.exists('ms_train/.ok'):
    tgz = telecharge(['https://openslr.elda.org/resources/108/FR.tgz', 'https://www.openslr.org/resources/108/FR.tgz',
                      'https://openslr.trmal.net/resources/108/FR.tgz'], 'dl/mediaspeech_FR.tgz')
    with tarfile.open(tgz) as t:
        for m in tqdm(t.getmembers()):
            if not (m.isfile() and m.name.lower().endswith(('.flac', '.wav'))): continue
            nom = os.path.basename(m.name).rsplit('.', 1)[0]
            dest = f"{'ms_test' if tenu_a_part(nom) else 'ms_train'}/{nom}.wav"
            with tempfile.NamedTemporaryFile(suffix=os.path.splitext(m.name)[1]) as tmp:
                tmp.write(t.extractfile(m).read()); tmp.flush()
                ecrit16k(dest, charge16k(tmp.name))
    os.remove(tgz)
    open('ms_train/.ok', 'w').write('ok')
print(len(os.listdir('ms_train')), 'extraits radio/télé (apprentissage),', len(os.listdir('ms_test')), '(note finale)')

SOURCES = [('fr_speech', 'negative_datasets/fr_speech'), ('ms_train', 'negative_datasets/fr_media'),
           ('near_words', 'negative_datasets/near_words')]
if P['boitier'] and len(glob.glob('boitier/neg_train/*.wav')) >= 20:
    SOURCES.append(('boitier/neg_train', 'negative_datasets/boitier'))
aug_neg = augmenteur(['fma_16k', 'audioset_16k'], ['mit_rirs'], duree=None, proba_fond=0.3, snr=(5, 20))
for source, cible in SOURCES:
    print(cible, spectrogrammes(source, cible, aug_neg))

In [ ]:
import os, sys, json, glob, shutil, tempfile
import numpy as np
sys.path.insert(0, '/content'); os.chdir('/content')
from mesange_v4 import *
P = json.load(open('/content/params.json'))   # écrit par la cellule 3
cache = Cache(P['drive'])

# 9. Les données de la note finale, jamais vues à l'entraînement : parole française lue (MLS, locuteurs de test,
#    ≈ 4 h), radio/télé (MediaSpeech, le 5e mis de côté), faux réveils du boîtier mis de côté, et le mot
#    (locuteurs réservés, propre et « sali »), plus les vrais appels de l'export s'il y en a.
import random, tarfile
from huggingface_hub import list_repo_files, hf_hub_download
EVAL_MLS_HEURES = 4.0
if not os.path.exists('eval_mls/.ok'):
    os.makedirs('eval_mls', exist_ok=True)
    fichiers = sorted(f for f in list_repo_files('facebook/multilingual_librispeech', repo_type='dataset')
                      if f.startswith('data/mls_french/test/audio/'))
    total = 0.0
    for f in fichiers:
        if total >= EVAL_MLS_HEURES * 3600: break
        p = hf_hub_download('facebook/multilingual_librispeech', f, repo_type='dataset', local_dir='mls_dl')
        with tarfile.open(p) as t:
            for m in t.getmembers():
                if m.isfile() and m.name.lower().endswith(('.flac', '.wav', '.opus')):
                    with tempfile.NamedTemporaryFile(suffix=os.path.splitext(m.name)[1]) as tmp:
                        tmp.write(t.extractfile(m).read()); tmp.flush(); x = charge16k(tmp.name)
                    ecrit16k(f"eval_mls/{os.path.basename(m.name).rsplit('.', 1)[0]}.wav", x); total += len(x) / RATE
        os.remove(p)
    open('eval_mls/.ok', 'w').write('ok')

if not os.path.exists('eval_positifs_bruit/.ok'):
    random.seed(7); np.random.seed(7)
    aug_eval = augmenteur(['fma_16k', 'audioset_16k'], ['mit_rirs'], snr=(0, 15), proba_fond=1.0)
    for f in sorted(glob.glob('eval_positifs/*.wav')):
        x = aug_eval.augment_clip(charge16k(f))
        ecrit16k(f"eval_positifs_bruit/{os.path.basename(f)}", np.concatenate([x, np.zeros(RATE // 2, np.float32)]))
    open('eval_positifs_bruit/.ok', 'w').write('ok')
for d in ['eval_mls', 'ms_test', 'eval_positifs', 'eval_positifs_bruit', 'boitier/neg_test', 'boitier/vrais']:
    print(d, len(glob.glob(d + '/*.wav')), 'fichiers')

In [ ]:
import os, sys, json, glob, shutil, tempfile
import numpy as np
sys.path.insert(0, '/content'); os.chdir('/content')
from mesange_v4 import *
P = json.load(open('/content/params.json'))   # écrit par la cellule 3
cache = Cache(P['drive'])

# 10. Réglages de l'entraînement (ceux de la v3, avec plus de poids pour le mot).
import yaml
def neg(d, poids, penalite=1.0):
    return {"features_dir": d, "sampling_weight": poids, "penalty_weight": penalite, "truth": False,
            "truncation_strategy": "random", "type": "mmap"}
features = [
    # v4 : 5 au lieu de 2, pour garder la part du mot de la recette microWakeWord (2 sur 27, 7 %) malgré les
    # contre-exemples français ajoutés (v3 : 2 sur 61, 3 %).
    {"features_dir": "generated_augmented_features", "sampling_weight": 5.0, "penalty_weight": 1.0, "truth": True,
     "truncation_strategy": "truncate_start", "type": "mmap"},
    neg("negative_datasets/speech", 10.0), neg("negative_datasets/dinner_party", 10.0), neg("negative_datasets/no_speech", 5.0),
    neg("negative_datasets/fr_speech", 12.0),                 # parole française lue
    neg("negative_datasets/fr_media", 12.0),                  # radio et télé françaises
    neg("negative_datasets/near_words", 4.0, 2.0),            # « mes anges », « mise en… »
]
if os.path.exists('negative_datasets/boitier/.ok'):
    features.append(neg("negative_datasets/boitier", 6.0, 3.0))   # les vrais faux réveils du boîtier
features.append({"features_dir": "negative_datasets/dinner_party_eval", "sampling_weight": 0.0, "penalty_weight": 1.0,
                 "truth": False, "truncation_strategy": "split", "type": "mmap"})   # validation seulement
config = {
    "window_step_ms": 10, "train_dir": "trained_models/wakeword", "features": features,
    "training_steps": [int(P['etapes'])], "positive_class_weight": [1], "negative_class_weight": [20],
    "learning_rates": [0.001], "batch_size": 128,
    "time_mask_max_size": [0], "time_mask_count": [0], "freq_mask_max_size": [0], "freq_mask_count": [0],
    "eval_step_interval": 500, "clip_duration_ms": 1500,
    "target_minimization": 0.9, "minimization_metric": None, "maximization_metric": "average_viable_recall",
}
yaml.dump(config, open('training_parameters.yaml', 'w'))
print([f['features_dir'] for f in features])

In [ ]:
import os, sys, json, glob, shutil, tempfile
import numpy as np
sys.path.insert(0, '/content'); os.chdir('/content')
from mesange_v4 import *
P = json.load(open('/content/params.json'))   # écrit par la cellule 3
cache = Cache(P['drive'])

# 11. L'entraînement (≈ 1 h 30 à 2 h sur T4). Il reprend à sa dernière étape gardée (toutes les 500 étapes,
#     copiée sur Drive) : après une coupure, relancer simplement les cellules depuis le début.
#     Puis la conversion en modèle quantifié pour le boîtier. « ETAPE n sur 15000 gardee » montre l'avancement.
TFLITE = 'trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite'
MIROIR = f"{P['drive']}/trained_models_wakeword" if P['drive'] else ''
if MIROIR and not os.path.exists('trained_models/wakeword/resume.json') and os.path.exists(f'{MIROIR}/resume.json'):
    miroir(MIROIR, 'trained_models/wakeword'); print('entraînement repris de Drive')
if os.path.exists('trained_models/wakeword/resume.json'):
    print('dernière étape gardée :', json.load(open('trained_models/wakeword/resume.json'))['step'])
fini = False
if os.path.exists(TFLITE) and os.path.exists('trained_models/wakeword/resume.json'):
    fini = json.load(open('trained_models/wakeword/resume.json'))['step'] >= int(P['etapes'])
if fini:
    print('Entraînement et conversion déjà faits.')
else:
    os.environ['MESANGE_MIROIR'] = MIROIR
    sh("python -m microwakeword.model_train_eval --training_config='training_parameters.yaml' --train 1 "
       "--restore_checkpoint 1 --test_tf_nonstreaming 0 --test_tflite_nonstreaming 0 --test_tflite_nonstreaming_quantized 0 "
       "--test_tflite_streaming 0 --test_tflite_streaming_quantized 1 --use_weights best_weights "
       "mixednet --pointwise_filters 64,64,64,64 --repeat_in_block 1,1,1,1 "
       "--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' --residual_connection 0,0,0,0 "
       "--first_conv_filters 32 --first_conv_kernel_size 5 --stride 3", flux=True)
    miroir('trained_models/wakeword', MIROIR)
assert os.path.exists(TFLITE), 'le modèle quantifié manque : voir la sortie au-dessus'
print('modèle :', TFLITE, os.path.getsize(TFLITE), 'octets')

In [ ]:
import os, sys, json, glob, shutil, tempfile
import numpy as np
sys.path.insert(0, '/content'); os.chdir('/content')
from mesange_v4 import *
P = json.load(open('/content/params.json'))   # écrit par la cellule 3
cache = Cache(P['drive'])

# 12. ÉVALUATION -- mesure de référence, NE PAS MODIFIER.
#     Le modèle est noté exactement comme le boîtier le fait tourner (frontend 40 bandes, int8, moyenne des
#     5 dernières sorties, 15 inférences ignorées après un réveil) :
#      - réveils par heure sur de la parole française jamais entendue (MLS test, radio/télé MediaSpeech mise de côté)
#        et sur les faux réveils du boîtier mis de côté ;
#      - rappel : part des « Mésange » de locuteurs jamais entendus qui réveillent (propre, et avec bruit/pièce),
#        et des vrais appels de l'export.
import time
TFLITE = 'trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite'
t0 = time.time()
modele = ModeleBoitier(TFLITE)
fa = {}
for nom, d in [('fr_mls', 'eval_mls'), ('fr_radio_tv', 'ms_test'), ('boitier', 'boitier/neg_test')]:
    fs = sorted(glob.glob(d + '/*.wav'))
    if not fs: continue
    pcm = flux_de_dossier(fs)
    fa[nom] = {'heures': len(pcm) / RATE / 3600, 'scores': scores_flux(modele, pcm)}
    print(f"{nom} : {fa[nom]['heures']:.2f} h")
rappels = {}
for nom, d in [('propre', 'eval_positifs'), ('bruit', 'eval_positifs_bruit'), ('vrais_appels', 'boitier/vrais')]:
    fs = sorted(glob.glob(d + '/*.wav'))
    if fs: rappels[nom] = [note_clip(modele, f) for f in fs]   # entouré de 0,5 s de silence
lignes = tableau_metriques(fa, rappels)
conseil = seuil_conseille(lignes)
metriques = {
    'modele': 'mesange v4', 'etapes': int(P['etapes']), 'date': time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime()),
    'heures': {k: round(v['heures'], 3) for k, v in fa.items()}, 'clips': {k: len(v) for k, v in rappels.items()},
    'seuils': lignes, 'seuil_conseille': conseil,
    'regle_seuil': 'au plus 0,5 réveil/h sur la parole française, puis le plus de vrais appels, le meilleur rappel « bruit », le plus bas ; sinon le moins de réveils',
    'export_boitier': P['boitier'], 'versions': versions(), 'duree_evaluation_s': round(time.time() - t0),
}
json.dump(metriques, open('metriques-v4.json', 'w'), indent=1, ensure_ascii=False)
print(texte_tableau(lignes))
print('seuil conseillé :', conseil)

In [ ]:
import os, sys, json, glob, shutil, tempfile
import numpy as np
sys.path.insert(0, '/content'); os.chdir('/content')
from mesange_v4 import *
P = json.load(open('/content/params.json'))   # écrit par la cellule 3
cache = Cache(P['drive'])

# 13. Le paquet final : mesange.tflite + mesange.json + metriques-v4.json dans mesange-v4.zip, téléchargé (et copié sur Drive).
import zipfile
TFLITE = 'trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite'
m = json.load(open('metriques-v4.json'))
manifeste = {'wake_word': P['mot'],   # même forme que le mesange.json de la v1
             'micro': {'probability_cutoff': m['seuil_conseille'], 'sliding_window_size': 5, 'feature_step_size': 10,
                       'tensor_arena_size': 30000, 'minimum_esphome_version': '2024.7.0'},
             'version': 2, 'type': 'micro', 'trained_languages': ['fr']}
os.makedirs('sortie', exist_ok=True)
shutil.copy(TFLITE, f"sortie/{P['nom']}.tflite")
json.dump(manifeste, open(f"sortie/{P['nom']}.json", 'w'), indent=2, ensure_ascii=False)
shutil.copy('metriques-v4.json', 'sortie/metriques-v4.json')
shutil.copy('training_parameters.yaml', 'sortie/training_parameters.yaml')
with zipfile.ZipFile('mesange-v4.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir('sortie')): z.write(f'sortie/{f}', f)
print(zipfile.ZipFile('mesange-v4.zip').namelist(), os.path.getsize('mesange-v4.zip'), 'octets')
if P['drive']:
    shutil.copy('mesange-v4.zip', f"{P['drive']}/mesange-v4.zip"); print('copié sur Drive :', f"{P['drive']}/mesange-v4.zip")
from google.colab import files
files.download('mesange-v4.zip')
print(texte_tableau(m['seuils']))
print('seuil conseillé :', m['seuil_conseille'])